# 보안프로그래밍 — 양자내성암호(PQC) 실습

**대상**: 학부 4학년 

본 노트북은 NIST가 표준화한 양자내성암호 3종(ML-KEM, ML-DSA, SLH-DSA)의 동작 원리를 직접 실행해보고, 그 구현·운용 과정에서 발생할 수 있는 대표적 **취약점을 코드로 재현**하는 것을 목표로 한다.

## 학습 목표
1. NIST FIPS 203/204/205 알고리즘을 Python으로 직접 호출해본다.
2. 고전 암호(X25519, Ed25519, ECDSA)와의 사이즈/성능 차이를 측정한다.
3. PQC 사용 시 발생할 수 있는 4가지 취약점을 직접 만들고 깨뜨려본다.
   - 잘못된 하이브리드 결합기
   - 타이밍 부채널 (KyberSlash 스타일)
   - 무작위성 실패 (논스 재사용)
   - 도메인 분리 누락
4. 안전한 하이브리드 PQ 핸드셰이크의 6가지 핵심 원칙을 설명할 수 있다.

## 진행 방식
- 위에서 아래로 셀을 차례대로 실행
- 🧪 표시 셀은 **직접 코드를 채워야** 한다
- ❓ 표시 셀은 답을 적거나 옆 사람과 토론한다

> ⚠️ **경고**: 본 노트북의 취약 코드는 **교육 목적**이다. 운영 시스템에 절대 사용하지 말 것. 운영에서는 검증된 라이브러리(libcrux, AWS-LC, BoringSSL)를 사용한다.


---
## §0. 환경 설정 (15분)

다음 라이브러리를 설치한다:

- **`kyber-py`**: NIST FIPS 203 (ML-KEM)의 **순수 Python 구현**. 내부 코드를 들여다볼 수 있어 교육용으로 적합.
- **`dilithium-py`**: NIST FIPS 204 (ML-DSA)의 순수 Python 구현.
- **`cryptography`**: X25519, Ed25519 등 고전 암호 비교용 (Colab 기본 포함).
- **`ecdsa`**: ECDSA 논스 재사용 데모용.

> 📌 운영에서는 `libcrux`(Rust, 형식 검증), `liboqs`, `AWS-LC` 등을 쓴다. 본 노트북의 순수 Python 구현은 **느리고 부채널에 취약**하다 — 그게 우리가 §5에서 시연할 포인트이기도 하다.


In [ ]:
!pip install -q kyber-py dilithium-py cryptography ecdsa

In [ ]:
import os
import time
import hashlib
import hmac
import secrets
from statistics import mean, median, stdev

# PQC (순수 Python 구현 — 내부 검사 가능)
from kyber_py.ml_kem import ML_KEM_512, ML_KEM_768, ML_KEM_1024
from dilithium_py.ml_dsa import ML_DSA_44, ML_DSA_65, ML_DSA_87

# 고전 암호 (비교용)
from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

print("✅ 라이브러리 로드 완료")
print()
print("NIST PQC 표준 (FIPS, 2024-08-13):")
print("  FIPS 203 — ML-KEM   (구 Kyber)     : 키 캡슐화")
print("  FIPS 204 — ML-DSA   (구 Dilithium) : 디지털 서명")
print("  FIPS 205 — SLH-DSA  (구 SPHINCS+)  : 디지털 서명 (해시 기반)")


---
## §1. ML-KEM (Kyber) — 키 캡슐화 (30분)

### 개념
**KEM(Key Encapsulation Mechanism)**은 Diffie-Hellman과 *목적*이 같지만 *모양*이 다르다.

- **DH**: 양쪽이 공개키를 교환하고 *함께* 공유 키를 계산
- **KEM**: 한쪽(B)이 공개키를 공개 → 다른 쪽(A)이 임의 키를 *캡슐화*해 보냄 → B가 *복호화*해서 같은 키 획득

```
A                                     B
                          ←  pk_B (encapsulation key)
(K, ct) = Encaps(pk_B)
                              ct  →
                                       K = Decaps(sk_B, ct)
```

ML-KEM은 **Module-LWE** 문제(격자 기반)에 안전성을 둔다. 양자컴퓨터로도 풀기 어렵다고 추정된다.

### 매개변수
| 변형 | NIST 보안 레벨 | 비고 |
|---|---|---|
| ML-KEM-512  | Level 1 (≈ AES-128) | 최저 |
| ML-KEM-768  | Level 3 (≈ AES-192) | **권장** (TLS 등) |
| ML-KEM-1024 | Level 5 (≈ AES-256) | 최고 보안 |


In [ ]:
# 1.1 ML-KEM-768 키 생성
ek, dk = ML_KEM_768.keygen()
#   ek = encapsulation key (공개키)
#   dk = decapsulation key (개인키)

print(f"ek (공개키)  크기: {len(ek):>5} bytes")
print(f"dk (개인키)  크기: {len(dk):>5} bytes")
print(f"ek 처음 32바이트: {ek[:32].hex()}")


In [ ]:
# 1.2 캡슐화 / 복호화
K_alice, ct = ML_KEM_768.encaps(ek)        # Alice가 Bob의 공개키로 캡슐화
K_bob = ML_KEM_768.decaps(dk, ct)          # Bob이 자기 개인키로 복호화

print(f"공유 키 K (32B): {K_alice.hex()}")
print(f"ciphertext 크기: {len(ct)} bytes")
print(f"양쪽 키 일치? {K_alice == K_bob}")


### ❓ 토론 1
- 공유 키 `K`가 **32바이트**인 이유는? (힌트: AES-256 대칭 키)
- ML-KEM-768의 ciphertext가 1088바이트인데, X25519의 32바이트와 비교했을 때 TLS 핸드셰이크에 어떤 영향이 있을까?


In [ ]:
# 1.3 사이즈 비교: 고전 vs PQ
print(f"{'알고리즘':<20}{'공개키':>10}{'개인키':>10}{'캡슐/서명':>12}")
print("-" * 52)

# 고전: X25519
x_priv = x25519.X25519PrivateKey.generate()
x_pub  = x_priv.public_key()
x_pub_bytes  = x_pub.public_bytes_raw()
x_priv_bytes = x_priv.private_bytes_raw()
print(f"{'X25519 (ECDH)':<20}{len(x_pub_bytes):>10}{len(x_priv_bytes):>10}{32:>12}")

# ML-KEM 3종
for name, alg in [("ML-KEM-512", ML_KEM_512),
                  ("ML-KEM-768", ML_KEM_768),
                  ("ML-KEM-1024", ML_KEM_1024)]:
    ek, dk = alg.keygen()
    _, ct = alg.encaps(ek)
    print(f"{name:<20}{len(ek):>10}{len(dk):>10}{len(ct):>12}")


In [ ]:
# 1.4 성능 비교: keygen + encaps + decaps 한 사이클
N = 30   # 측정 횟수 (순수 Python 구현이라 느리니 작게)

def bench(label, fn, n=N):
    times = []
    for _ in range(n):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)  # ms
    print(f"{label:<20} median={median(times):8.3f} ms   stdev={stdev(times):6.3f}")

def x25519_cycle():
    a = x25519.X25519PrivateKey.generate()
    b = x25519.X25519PrivateKey.generate()
    a.exchange(b.public_key())

def mlkem768_cycle():
    ek, dk = ML_KEM_768.keygen()
    K, ct = ML_KEM_768.encaps(ek)
    ML_KEM_768.decaps(dk, ct)

bench("X25519 ECDH 1회",  x25519_cycle)
bench("ML-KEM-768 1사이클", mlkem768_cycle)

print()
print("⚠️  kyber-py는 순수 Python이라 매우 느림.")
print("   C/Rust 구현(libcrux, liboqs)에서는 ML-KEM이 X25519와 비슷하거나 더 빠름.")


### 🧪 실습 1
아래 빈칸을 채워, **A → B → A의 양방향 KEM 핸드셰이크**를 구현하라.
- A가 자기 KEM 공개키를 B에게 보냄
- B가 캡슐화한 ct를 A에게 회신
- 양쪽이 같은 32바이트 공유 키를 도출


In [ ]:
# 🧪 실습 1: 양방향 KEM 핸드셰이크
# TODO 부분을 채우시오

# A 측
ek_A, dk_A = ML_KEM_768.keygen()

# 네트워크 전송 시뮬레이션 (직렬화 없이 바이트 그대로 전송)
ek_received_by_B = ek_A

# B 측
K_B, ct = ML_KEM_768.encaps(ek_received_by_B)

# 다시 A로 회신
ct_received_by_A = ct
K_A = None  # TODO: A는 자기 dk_A로 ct를 decaps 하시오

# 검증
assert K_A == K_B, "양쪽 공유 키가 다름!"
print("✅ 양방향 KEM 성공! 공유 키:", K_A.hex())


---
## §2. ML-DSA (Dilithium) — 디지털 서명 (25분)

### 개념
ML-DSA는 ML-KEM과 같은 **격자(MLWE) 기반** 서명 스킴이다. 구조는 **Fiat–Shamir with Aborts** 패턴을 사용하며 실패 시 다시 샘플링한다.

### 매개변수
| 변형 | NIST 레벨 | 권장 사용처 |
|---|---|---|
| ML-DSA-44 | Level 2 | 일반 |
| ML-DSA-65 | Level 3 | **권장** |
| ML-DSA-87 | Level 5 | 최고 보안 |

### 핵심 API
```python
pk, sk = ML_DSA_65.keygen()
sig = ML_DSA_65.sign(sk, msg, ctx=b"")             # 서명
ok  = ML_DSA_65.verify(pk, msg, sig, ctx=b"")      # 검증
```

`ctx`는 **도메인 분리 컨텍스트**다. §7에서 다룬다.


In [ ]:
# 2.1 ML-DSA-65 서명 / 검증
pk, sk = ML_DSA_65.keygen()

msg = b"Transfer 1000 KRW from Alice to Bob"
sig = ML_DSA_65.sign(sk, msg)

print(f"공개키 크기: {len(pk):>5} bytes")
print(f"개인키 크기: {len(sk):>5} bytes")
print(f"서명 크기  : {len(sig):>5} bytes")
print(f"검증 결과  : {ML_DSA_65.verify(pk, msg, sig)}")


In [ ]:
# 2.2 메시지 변조 → 검증 실패 확인
tampered = b"Transfer 9999 KRW from Alice to Bob"
print("원본 검증     :", ML_DSA_65.verify(pk, msg, sig))
print("변조본 검증   :", ML_DSA_65.verify(pk, tampered, sig))   # False여야 함

# 서명 1바이트 뒤집기
sig_bad = bytearray(sig); sig_bad[0] ^= 0x01
print("서명변조 검증 :", ML_DSA_65.verify(pk, msg, bytes(sig_bad)))


In [ ]:
# 2.3 사이즈 비교: Ed25519 vs ML-DSA
print(f"{'알고리즘':<20}{'공개키':>10}{'개인키':>10}{'서명':>10}")
print("-" * 50)

# Ed25519
e_priv = ed25519.Ed25519PrivateKey.generate()
e_pub  = e_priv.public_key()
e_sig  = e_priv.sign(msg)
print(f"{'Ed25519':<20}{32:>10}{32:>10}{len(e_sig):>10}")

# ML-DSA 3종
for name, alg in [("ML-DSA-44", ML_DSA_44),
                  ("ML-DSA-65", ML_DSA_65),
                  ("ML-DSA-87", ML_DSA_87)]:
    pk, sk = alg.keygen()
    sig = alg.sign(sk, msg)
    print(f"{name:<20}{len(pk):>10}{len(sk):>10}{len(sig):>10}")


### ❓ 토론 2

- **코드 서명** 시나리오: 100만 개의 RPM 패키지에 서명한다. ML-DSA-65로 바꾸면 추가 디스크 사용량은?
- **TLS 인증서 체인**: 보통 3장이 들어간다. 각 인증서에 ML-DSA-65 공개키 + 서명이 포함되면 핸드셰이크당 추가 바이트는?


---
## §3. SLH-DSA + 종합 비교 (15분)

### SLH-DSA (구 SPHINCS+)
- **해시 기반 서명**. Merkle 트리 + WOTS+ + FORS의 조합.
- 보안 가정이 가장 단순함: **해시 함수가 안전하면 OK**. 격자보다 보수적.
- 단점: **서명 사이즈가 매우 큼 (8 KB ~ 50 KB)**, 서명 시간 매우 김.
- 용도: 격자가 깨지는 최악의 시나리오 대비. 펌웨어/루트 신뢰 같은 *드물게 서명, 자주 검증* 환경.

본 강의에서는 코드 데모 대신 **알고리즘 종합 비교 표**로 갈음한다 (순수 Python SLH-DSA가 너무 느려 실시간 데모가 어려움).


In [ ]:
# 3.1 알고리즘 종합 비교 (NIST 보안 레벨 3 기준)
print(f"{'알고리즘':<22}{'pk':>8}{'sk':>8}{'ct/sig':>10}{'기반':>15}{'양자안전?':>12}")
print("-" * 75)

table = [
    ("X25519 (ECDH)",        32,    32,    32,    "이산로그",   "❌"),
    ("Ed25519",              32,    32,    64,    "이산로그",   "❌"),
    ("ECDSA P-256",          64,    32,    64,    "이산로그",   "❌"),
    ("RSA-3072",            384,  3072,   384,    "인수분해",   "❌"),
    ("---", 0, 0, 0, "", ""),
    ("ML-KEM-768",         1184,  2400,  1088,    "격자(MLWE)",  "✅"),
    ("ML-DSA-65",          1952,  4032,  3309,    "격자(MLWE)",  "✅"),
    ("SLH-DSA-192s (예상)",   48,    96,  16224,   "해시",       "✅"),
    ("HQC-192 (예상)",     4522,  4562,  9026,    "코드기반",    "✅"),
]
for row in table:
    if row[0] == "---":
        print("-" * 75); continue
    print(f"{row[0]:<22}{row[1]:>8}{row[2]:>8}{row[3]:>10}{row[4]:>15}{row[5]:>12}")


### 핵심 관찰
1. **사이즈는 모두 커진다.** 특히 RSA보다도 ML-DSA 공개키가 작은 건 흥미롭다.
2. **속도**: ML-KEM/ML-DSA는 ECC 수준. SLH-DSA는 매우 느림.
3. **HQC**는 ML-KEM 백업으로 표준화 중 — 구조가 다른 두 알고리즘을 가지면 한 가족이 깨져도 대비 가능.
4. **개인키 압축**: ML-KEM/ML-DSA 모두 32~64 바이트 시드로 재생성 가능.


---
## §4. 취약점 ①: 하이브리드 결합기 오류 (25분)

### 배경
PQC 마이그레이션 초기에는 **고전 + PQ 하이브리드**를 많이 쓴다 (NIST IR 8547):
> "두 구성 요소 중 *하나만 안전*해도 전체가 안전해야 한다."

핵심은 **두 공유 키를 어떻게 결합하느냐**이다. 잘못된 결합기는 하이브리드의 안전 성질을 날려버린다.

### 시나리오
- TLS-like 핸드셰이크
- 양쪽이 X25519 공유키 `K_x`와 ML-KEM 공유키 `K_p`를 도출
- 최종 세션 키 `K`를 결합기로 만든다


In [ ]:
# 4.1 두 가지 결합기 정의

def bad_combiner_xor_session(K_x: bytes, K_p: bytes, session_key: bytes):
    """
    나쁜 결합기: 세션 키를 두 채널로 따로 래핑
      c1 = session_key XOR K_x
      c2 = session_key XOR K_p
    겉보기엔 두 채널을 다 깨야 할 것 같지만 실은 *하나*만 깨도 됨.
    """
    c1 = bytes(a ^ b for a, b in zip(session_key, K_x))
    c2 = bytes(a ^ b for a, b in zip(session_key, K_p))
    return c1, c2

def good_combiner_kdf(K_x: bytes, K_p: bytes, transcript: bytes) -> bytes:
    """
    좋은 결합기: HKDF로 *연결*해서 KDF에 통과
    K = HKDF(K_x || K_p, transcript)
    두 입력 모두 알아야 K를 복원 가능.
    """
    ikm = K_x + K_p
    hkdf = HKDF(algorithm=hashes.SHA256(), length=32,
                salt=b"hybrid-v1", info=transcript)
    return hkdf.derive(ikm)

# 데모용 키들
K_x = secrets.token_bytes(32)        # X25519 공유키 (가정)
K_p = secrets.token_bytes(32)        # ML-KEM 공유키 (가정)
session_key = secrets.token_bytes(32)
transcript = b"client_hello||server_hello||..."

c1, c2 = bad_combiner_xor_session(K_x, K_p, session_key)
K_good = good_combiner_kdf(K_x, K_p, transcript)

print("나쁜 결합기 (XOR 분리):")
print(f"  c1 = {c1.hex()}")
print(f"  c2 = {c2.hex()}")
print(f"좋은 결합기 (HKDF 연결):")
print(f"  K  = {K_good.hex()}")


In [ ]:
# 4.2 [공격 시연] 미래에 X25519가 양자컴퓨터로 깨졌다고 가정
# 즉, 공격자는 K_x를 알게 됨

print("🦹 공격자 시나리오: 미래에 X25519의 K_x가 노출되었다")
print(f"   K_x (노출됨): {K_x.hex()}")
print()

# 나쁜 결합기에 대한 공격
recovered = bytes(a ^ b for a, b in zip(c1, K_x))
print("[나쁜 결합기 공격]")
print(f"  공격자가 복원한 session_key : {recovered.hex()}")
print(f"  실제 session_key             : {session_key.hex()}")
print(f"  ❌ 일치? {recovered == session_key}  →  PQ 부분이 무력화됨!")
print()

# 좋은 결합기는 K_p 없이 복원 불가
print("[좋은 결합기 공격]")
print("  공격자는 K_x만 알고 있음. K_p 없이는 HKDF 출력 계산 불가.")
print("  ✅ 하이브리드 안전성 유지: PQ 부분이 살아있는 한 K_good은 안전")


### 🔑 핵심 교훈

| 구분 | 나쁜 결합기 (XOR 래핑) | 좋은 결합기 (HKDF 연결) |
|---|---|---|
| 모양 | `c1 = K ⊕ K_x`, `c2 = K ⊕ K_p` | `K = HKDF(K_x ‖ K_p ‖ ctx)` |
| 안전 조건 | **둘 다** 안전해야 함 | **하나만** 안전해도 됨 |
| 부채널 | XOR 자체가 보장하는 게 적음 | KDF가 분포 평탄화 |

**X-Wing**(Cloudflare 등) 명세는 KDF에 **공개키, ciphertext, 두 공유 키** 모두 묶어 넣는 더 견고한 구조다. 운영 코드에서는 X-Wing 같은 표준을 쓰자.

### 🧪 실습 4
아래 코드는 **부분적으로 잘못된** 결합기다. 어떤 점이 위험한지 찾아 토론하라.


In [ ]:
# 🧪 실습 4: 이 결합기의 문제점은?

def suspicious_combiner(K_x: bytes, K_p: bytes) -> bytes:
    # transcript 없음! 또한 길이 분리(length separation)도 없음
    return hashlib.sha256(K_x + K_p).digest()

# Q1: transcript가 없으면 어떤 공격이 가능한가?
#     힌트: 동일 (K_x, K_p)로 두 다른 세션을 묶으면?
# Q2: 길이 분리(length separation)가 없으면?
#     힌트: K_x = b"AB", K_p = b"CD"  vs  K_x = b"ABC", K_p = b"D" 의 결과를 비교

print("길이 분리 부재 데모:")
print("  K_x=AB,  K_p=CD :", hashlib.sha256(b"AB" + b"CD").hexdigest())
print("  K_x=ABC, K_p=D  :", hashlib.sha256(b"ABC" + b"D").hexdigest())
print("  → 결과 동일! 다른 입력이 같은 키를 만들 수 있음 (충돌 유도 가능)")


---
## §5. 취약점 ②: 타이밍 부채널 — KyberSlash 스타일 (25분)

### 배경
**KyberSlash** (2024) — Kyber/ML-KEM의 일부 구현에서 **나눗셈 연산이 비밀 데이터에 의존하는 시간**을 갖던 취약점. 원격에서 비밀 키 복원이 가능했다.

원리는 단순하다:
1. 비밀 키와 관련된 값으로 분기/나눗셈을 한다
2. 입력에 따라 실행 시간이 달라진다
3. 공격자가 시간을 측정해 비밀을 추론한다

본 절에서는 **단순화된 두 가지 타이밍 누설**을 직접 측정해본다.

### 데모 1: 비-상수 시간 비교 함수


In [ ]:
# 5.1 두 가지 비교 함수
def vulnerable_compare(a: bytes, b: bytes) -> bool:
    """조기 종료(early exit)로 타이밍 누설"""
    if len(a) != len(b):
        return False
    for x, y in zip(a, b):
        if x != y:
            return False    # 첫 다른 바이트에서 즉시 종료 → 시간 차
    return True

def constant_time_compare(a: bytes, b: bytes) -> bool:
    """항상 모든 바이트를 보고 누적 OR"""
    if len(a) != len(b):
        return False
    result = 0
    for x, y in zip(a, b):
        result |= (x ^ y)
    return result == 0

# 일치 검증
print(vulnerable_compare(b"abc", b"abc"), constant_time_compare(b"abc", b"abc"))
print(vulnerable_compare(b"abc", b"abd"), constant_time_compare(b"abc", b"abd"))


In [ ]:
# 5.2 타이밍 측정: 처음 N바이트가 일치할 때 시간이 길어지는가?
import statistics

SECRET = bytes([0x42] * 64)   # 비밀 (서버 측에 있다고 가정)

def measure(target: bytes, fn, repeat=2000):
    times = []
    for _ in range(repeat):
        t = time.perf_counter_ns()
        fn(SECRET, target)
        times.append(time.perf_counter_ns() - t)
    return statistics.median(times)

print(f"{'일치 prefix':<15}{'vulnerable (ns)':>20}{'constant-time (ns)':>22}")
print("-" * 57)
for prefix_len in [0, 1, 8, 16, 32, 48, 60, 64]:
    guess = bytes([0x42] * prefix_len) + bytes([0xFF] * (64 - prefix_len))
    t_vuln  = measure(guess, vulnerable_compare)
    t_const = measure(guess, constant_time_compare)
    print(f"{prefix_len:<15}{t_vuln:>20}{t_const:>22}")


prefix 일치가 길어질수록 `vulnerable_compare`의 시간이 *증가*해야 한다 (Python 인터프리터 잡음으로 가끔 묻힐 수 있다 — C 구현에서는 훨씬 명확하다).

이 누설을 활용한 **바이트 단위 점진적 추측 공격**:
1. 첫 바이트를 0~255로 시도하며 가장 느린 값 = 일치
2. 그 다음 바이트를 같은 방식으로 …
3. 256 × 64 ≈ 16,000회 측정으로 64바이트 비밀 복원

### 데모 2: KyberSlash 스타일 — 데이터 의존 분기


In [ ]:
# 5.3 KyberSlash 풍 데이터 의존 분기
def vulnerable_compress(secret_coeff: int, q: int = 3329) -> int:
    """
    가상의 비-상수 시간 압축. 비밀 계수가 클수록 루프가 더 돈다.
    실제 KyberSlash는 컴파일러가 생성한 나눗셈 명령에서 이런 효과가 있었다.
    """
    while secret_coeff >= q:
        secret_coeff -= q
    return secret_coeff

def constant_time_compress(secret_coeff: int, q: int = 3329) -> int:
    """항상 동일한 연산 횟수"""
    return secret_coeff % q

small_input = 100
large_input = 3329 * 1000 + 5     # q의 1000배 + 5

def time_fn(fn, x, n=50000):
    t = time.perf_counter_ns()
    for _ in range(n):
        fn(x)
    return (time.perf_counter_ns() - t) / n

print("vulnerable_compress:")
print(f"  small input ({small_input}):       {time_fn(vulnerable_compress, small_input):.1f} ns/op")
print(f"  large input (~{large_input}):  {time_fn(vulnerable_compress, large_input):.1f} ns/op")
print()
print("constant_time_compress:")
print(f"  small input  : {time_fn(constant_time_compress, small_input):.1f} ns/op")
print(f"  large input  : {time_fn(constant_time_compress, large_input):.1f} ns/op")
print()
print("→ vulnerable 버전은 입력 크기 = 비밀에 따라 시간이 변한다.")


### 🔑 핵심 교훈

1. **상수 시간 코드** = 모든 입력에 대해 *동일한 연산 시퀀스*.
   - 비밀에 의존한 분기 ❌
   - 비밀 인덱스로 배열 접근 ❌ (캐시 타이밍)
   - 비밀에 의존한 나눗셈 ❌ (CPU별로 가변)

2. **Python에서는 진짜 상수 시간 코드를 쓰기 어렵다** — 인터프리터의 정수 표현, GC 등이 모두 비-상수 시간. 그래서 PQC는 **C/Rust + 검증**이 표준이다.

3. **`hmac.compare_digest`** 또는 OpenSSL `CRYPTO_memcmp` 같은 검증된 비교 함수만 사용한다.

### 🧪 실습 5
아래 검증 함수를 **상수 시간**으로 고쳐라.


In [ ]:
# 🧪 실습 5: 상수 시간으로 고치기

def verify_mac_BAD(received_mac: bytes, expected_mac: bytes) -> bool:
    """이 함수는 타이밍에 취약하다. 왜?"""
    return received_mac == expected_mac    # Python의 == 는 조기 종료할 수 있음

def verify_mac_GOOD(received_mac: bytes, expected_mac: bytes) -> bool:
    # TODO: hmac.compare_digest 또는 위의 constant_time_compare 사용
    return hmac.compare_digest(received_mac, expected_mac)

# 검증
m1 = b"\x00" * 32
m2 = b"\x00" * 32
m3 = b"\xff" * 32
assert verify_mac_GOOD(m1, m2) == True
assert verify_mac_GOOD(m1, m3) == False
print("✅ verify_mac_GOOD 통과")


---
## §6. 취약점 ③: 무작위성과 결정성 (25분)

### 배경
서명 스킴에서 **랜덤 논스(nonce)**의 잘못된 처리는 *역대급* 사건들을 만들었다:
- **Sony PS3 (2010)**: ECDSA 논스를 상수로 사용 → 마스터 키 유출
- **Bitcoin Android wallet (2013)**: SecureRandom 버그로 같은 k 재사용 → 키 도난
- **Debian OpenSSL (2008)**: 엔트로피 풀 손상 → 수년간의 키 모두 추측 가능

### ECDSA 논스 재사용 공격 (수학)
같은 키 `d`, 같은 nonce `k`로 두 메시지에 서명:
```
s1 = k⁻¹ (h1 + r·d)  mod n
s2 = k⁻¹ (h2 + r·d)  mod n      (같은 r은 같은 k에서 나오므로)

s1 - s2 = k⁻¹ (h1 - h2)  mod n
→ k = (h1 - h2) / (s1 - s2)  mod n
→ d = (s1·k - h1) / r        mod n
```


In [ ]:
# 6.1 ECDSA 논스 재사용으로 개인키 복원
from ecdsa import SigningKey, VerifyingKey, NIST256p
from ecdsa.util import sigdecode_string, string_to_number
from ecdsa.numbertheory import inverse_mod

# 공격자에게 노출된 정보: 두 (msg, sig) 쌍 — 같은 k 사용
sk = SigningKey.generate(curve=NIST256p)
pk = sk.get_verifying_key()
n = NIST256p.order

REUSED_K = 0xC0FFEE_DEADBEEF_C0FFEE_DEADBEEF_C0FFEE_DEADBEEF_C0FFEE_DEADBEEF
m1 = b"Pay 100 to Alice"
m2 = b"Pay 100 to Bob"

sig1 = sk.sign(m1, k=REUSED_K)
sig2 = sk.sign(m2, k=REUSED_K)

# 공격자 단계
r1, s1 = sigdecode_string(sig1, n)
r2, s2 = sigdecode_string(sig2, n)
print(f"r1 == r2 ? {r1 == r2}    ← 같은 k의 결정적 신호")

h1 = string_to_number(hashlib.sha1(m1).digest())   # ecdsa 기본은 SHA-1
h2 = string_to_number(hashlib.sha1(m2).digest())

# k 복원
k = ((h1 - h2) * inverse_mod(s1 - s2, n)) % n
# d 복원
d_recovered = ((s1 * k - h1) * inverse_mod(r1, n)) % n

# 진짜 개인키와 비교
d_real = sk.privkey.secret_multiplier
print(f"복원된 k        : {hex(k)}")
print(f"복원된 d        : {hex(d_recovered)[:32]}...")
print(f"실제 d          : {hex(d_real)[:32]}...")
print(f"🦹 키 복원 성공? {d_recovered == d_real}")


### ML-DSA의 방어책 — Hedged 서명
ML-DSA 표준은 두 모드를 제공한다:

1. **Randomized (기본)**: `rnd ← random(32)`을 매 서명마다 새로 뽑음
2. **Deterministic**: `rnd = 0` 으로 고정 → 같은 메시지에 항상 같은 서명

NIST FIPS 204의 **권장 모드**는 **hedged**이다: `rnd`를 RNG에서 뽑되, 내부적으로 메시지와도 섞는다. 따라서:
- RNG가 망가져도 (rnd가 늘 같아도) 메시지가 다르면 서명이 다름
- 같은 (메시지, rnd) 쌍이 다시 와도 결정적 서명 → 안전


In [ ]:
# 6.2 ML-DSA 결정성 / 무작위성 모드 비교
pk, sk = ML_DSA_65.keygen()
m = b"hello world"

# Randomized (기본): 두 번 서명 → 다른 결과
s1 = ML_DSA_65.sign(sk, m, deterministic=False)
s2 = ML_DSA_65.sign(sk, m, deterministic=False)
print(f"Randomized — 두 서명 동일? {s1 == s2}")

# Deterministic: 두 번 서명 → 같은 결과
s3 = ML_DSA_65.sign(sk, m, deterministic=True)
s4 = ML_DSA_65.sign(sk, m, deterministic=True)
print(f"Deterministic — 두 서명 동일? {s3 == s4}")

# 둘 다 검증은 통과
assert ML_DSA_65.verify(pk, m, s1)
assert ML_DSA_65.verify(pk, m, s3)
print("✅ 두 모드 모두 검증 통과")


### ❓ 토론 6
- ECDSA에서 같은 k가 두 번 쓰이면 키가 나오는데, **ML-DSA에서 같은 `rnd`가 두 번 쓰이면** 같은 사고가 나는가?
- 답: 아니오. ML-DSA는 메시지를 `rnd`와 함께 해시해서 내부 무작위성을 만든다. 다른 메시지면 다른 내부 무작위성 → 다른 서명.
- 다만 **같은 (msg, rnd) 쌍**이 *다른 키*로 서명된 두 인스턴스가 있으면? 부채널 분석 측면에서 다른 위험 가능.

### 🧪 실습 6
아래 ECDSA 서명자는 안전한가? 위험을 식별하고 두 가지 이상의 결함을 지적하라.


In [ ]:
# 🧪 실습 6: 이 ECDSA 서명자의 결함은?
import random

class WeakSigner:
    def __init__(self):
        self.sk = SigningKey.generate(curve=NIST256p)
        self.counter = 0

    def sign(self, msg: bytes) -> bytes:
        # 결함 1: random.Random은 암호용 RNG가 아니다 (예측 가능)
        # 결함 2: 카운터로 시드 → 동일 인스턴스 재시작 시 같은 k
        # 결함 3: k의 비트 수가 작음 → 격자 공격으로 단일 서명에서도 키 복원
        rng = random.Random(self.counter)
        self.counter += 1
        k = rng.randint(1, 2**32)            # n에 비해 너무 작음
        return self.sk.sign(msg, k=k)

# 결함 정리:
# - 비암호 RNG (Mersenne Twister는 출력 624개로 상태 복원 가능)
# - 예측 가능한 시드 (0, 1, 2, ...)
# - k의 사이즈가 너무 작음 (Bleichenbacher 격자 공격)
print("결함 3개 식별 완료")


---
## §7. 취약점 ④: 도메인 분리 누락 (15분)

### 배경
같은 키로 *서로 다른 용도*에 서명할 때, 한 용도의 서명이 다른 용도에 **재사용**되면 위조다. 이를 막는 게 **도메인 분리(domain separation)**.

ML-DSA의 `sign(sk, msg, ctx=...)`의 `ctx`가 그 역할을 한다.


In [ ]:
# 7.1 도메인 분리 누락 데모
pk, sk = ML_DSA_65.keygen()

# 같은 회사의 두 시스템이 같은 키 쌍을 공유한다고 가정
m_app = b"USER:alice|ACTION:agree-to-tos"
m_pay = b"USER:alice|ACTION:agree-to-tos"   # 우연히 같은 형식

# 컨텍스트 없이 서명
sig_no_ctx = ML_DSA_65.sign(sk, m_app, ctx=b"")

print("ctx 없을 때:")
print(f"  A에서 검증: {ML_DSA_65.verify(pk, m_app, sig_no_ctx, ctx=b'')}")
print(f"  B에서 검증: {ML_DSA_65.verify(pk, m_pay, sig_no_ctx, ctx=b'')}")
print("  → 같은 메시지면 둘 다 통과: cross-protocol replay 가능")
print()

# 컨텍스트로 도메인 분리
sig_app = ML_DSA_65.sign(sk, m_app, ctx=b"app/v1/push")
sig_pay = ML_DSA_65.sign(sk, m_pay, ctx=b"pay/v1/transfer")

print("ctx 사용 시:")
print(f"  A 서명을 A에서 검증 (올바른 ctx): {ML_DSA_65.verify(pk, m_app, sig_app, ctx=b'app/v1/push')}")
print(f"  A 서명을 B에서 검증 (잘못된 ctx): {ML_DSA_65.verify(pk, m_app, sig_app, ctx=b'pay/v1/transfer')}")
print("  → 컨텍스트가 다르면 검증 실패: cross-protocol 공격 차단")


### 🔑 컨텍스트 문자열 작성 규칙

1. **앱 식별자 + 버전 + 용도**: `b"myapp/v2/login-challenge"`
2. **버전 비트가 있어야** 향후 포맷 변경 시 안전
3. **문서화**: 어떤 컨텍스트가 어디서 쓰이는지 표로 관리
4. **고정 도메인**: 사용자 입력을 컨텍스트에 넣지 말 것 (인젝션 위험)

### ❓ 토론 7
- 회사 PKI에서 *코드 서명용* 키와 *문서 서명용* 키를 같은 ML-DSA 쌍으로 쓰는 게 가능한가? 컨텍스트만 잘 분리하면? 위험은?
- TLS 1.3에서 도메인 분리는 어떻게 이루어지는가? (힌트: `transcript hash`)


---
## §8. 종합 챌린지: 안전한 PQ 핸드셰이크 구현 (20분)

지금까지 배운 모든 것을 합쳐 **올바른 하이브리드 PQ 핸드셰이크**를 작성한다.

### 요구사항
1. **하이브리드**: X25519 + ML-KEM-768
2. **결합기**: HKDF로 두 공유 키 + 트랜스크립트 결합
3. **인증**: 서버는 ML-DSA-65로 트랜스크립트에 서명
4. **도메인 분리**: 모든 KDF/서명에 컨텍스트 문자열 포함
5. **상수 시간 비교**: 클라이언트의 인증 검증
6. **랜덤성**: 모든 ephemeral 값은 안전한 RNG


In [ ]:
# 8.1 안전한 하이브리드 핸드셰이크 — 참조 구현

CTX_KDF = b"pq-handshake/v1/session-key"
CTX_SIG = b"pq-handshake/v1/server-auth"

class Server:
    def __init__(self):
        self.sig_pk, self.sig_sk = ML_DSA_65.keygen()

    def handle_hello(self, client_x_pub_bytes: bytes, client_kem_pub: bytes):
        # 1) X25519: 서버측 ephemeral
        x_priv = x25519.X25519PrivateKey.generate()
        x_pub = x_priv.public_key().public_bytes_raw()
        client_x_pub = x25519.X25519PublicKey.from_public_bytes(client_x_pub_bytes)
        K_x = x_priv.exchange(client_x_pub)

        # 2) ML-KEM 캡슐화
        K_p, kem_ct = ML_KEM_768.encaps(client_kem_pub)

        # 3) 트랜스크립트
        transcript = hashlib.sha256(
            client_x_pub_bytes + client_kem_pub + x_pub + kem_ct
        ).digest()

        # 4) 결합 — HKDF, ctx 포함
        K_session = HKDF(
            algorithm=hashes.SHA256(), length=32,
            salt=b"hybrid-pq-v1", info=CTX_KDF + transcript
        ).derive(K_x + K_p)

        # 5) 서버 인증: 트랜스크립트에 서명 (ctx 포함)
        sig = ML_DSA_65.sign(self.sig_sk, transcript, ctx=CTX_SIG)

        return x_pub, kem_ct, sig, K_session

class Client:
    def __init__(self, server_sig_pk: bytes):
        self.server_sig_pk = server_sig_pk

    def hello(self):
        self.x_priv = x25519.X25519PrivateKey.generate()
        self.x_pub = self.x_priv.public_key().public_bytes_raw()
        self.kem_ek, self.kem_dk = ML_KEM_768.keygen()
        return self.x_pub, self.kem_ek

    def finish(self, server_x_pub: bytes, kem_ct: bytes, server_sig: bytes):
        s_x_pub = x25519.X25519PublicKey.from_public_bytes(server_x_pub)
        K_x = self.x_priv.exchange(s_x_pub)
        K_p = ML_KEM_768.decaps(self.kem_dk, kem_ct)

        transcript = hashlib.sha256(
            self.x_pub + self.kem_ek + server_x_pub + kem_ct
        ).digest()

        K_session = HKDF(
            algorithm=hashes.SHA256(), length=32,
            salt=b"hybrid-pq-v1", info=CTX_KDF + transcript
        ).derive(K_x + K_p)

        # 서버 서명 검증 (도메인 분리 ctx 사용)
        ok = ML_DSA_65.verify(self.server_sig_pk, transcript, server_sig, ctx=CTX_SIG)
        if not ok:
            raise ValueError("서버 인증 실패!")
        return K_session

# 핸드셰이크 실행
srv = Server()
cli = Client(srv.sig_pk)

x_c, kem_c = cli.hello()
x_s, kem_ct, sig, K_srv = srv.handle_hello(x_c, kem_c)
K_cli = cli.finish(x_s, kem_ct, sig)

print(f"양쪽 세션 키 일치? {K_cli == K_srv}")
print(f"세션 키: {K_cli.hex()}")


### 🧪 종합 실습 8
다음 *각각의* 변경을 가했을 때 어떤 결과가 나오는지 확인하고 왜 그런지 설명하라.

1. `CTX_SIG`를 양쪽에서 다르게 설정 (서버: `b"v1"`, 클라이언트: `b"v2"`)
2. `K_x + K_p` 대신 `K_x` 만 HKDF에 넣음 (PQ 부분 누락)
3. transcript 계산에서 `kem_ct`를 빼버림
4. 클라이언트가 `verify` 결과를 `==` 로 비교 (상수 시간 아님) — 어떤 위험?
5. Client의 `x25519.X25519PrivateKey.generate()` 대신 고정 시드 사용


---
## 마무리

### 오늘 배운 것
1. **NIST PQC 표준 3종**: ML-KEM, ML-DSA, SLH-DSA — 직접 호출하고 사이즈/속도 측정.
2. **고전 vs PQ**: 키와 서명 사이즈가 10~100배 커진다. 속도는 ML-* 계열이 ECC 수준.
3. **PQC 사용의 함정**:
   - 결합기 설계: KDF 기반 + 트랜스크립트 + 컨텍스트
   - 부채널: 비밀 의존 분기/나눗셈/메모리 접근 금지
   - 무작위성: 검증된 RNG, hedged 서명 모드
   - 도메인 분리: 모든 서명에 ctx, 모든 KDF에 info

### 코드 작성 시 체크리스트
- [ ] 직접 구현하지 않고 검증된 라이브러리(libcrux, AWS-LC, BoringSSL) 사용
- [ ] 알고리즘이 설정값으로 교체 가능한 구조 (crypto-agility)
- [ ] 모든 비교는 `hmac.compare_digest` 또는 동급 상수 시간 함수
- [ ] 모든 서명에 도메인 분리 컨텍스트
- [ ] 하이브리드 결합은 KDF로, 트랜스크립트 포함
- [ ] 키 시드는 메모리에서 명시적 삭제 (zeroize)
- [ ] 사이즈 가정 제거: 공개키 = 32B 같은 매직 넘버 없애기

### 참고 자료
- NIST FIPS 203, 204, 205 (https://csrc.nist.gov/projects/post-quantum-cryptography)
- Filippo Valsorda, *On 128 bit security against quantum attackers* — words.filippo.io/128-bits/
- KyberSlash — kyberslash.cr.yp.to
- libcrux (Rust 형식 검증 PQC) — github.com/cryspen/libcrux
- github.com/veorq/awesome-post-quantum


